# Databricks RAGアプリチュートリアル - 2. RAGエージェントの構築（Databricks Apps 方式）

このノートブックでは、`1_...` で作成した AI Search Index を活用する **Tool-calling RAG エージェント**を構築し、**Databricks Apps** にデプロイします。

## 2つのデプロイ方式

Databricks でエージェントをサービングする方式は2つあります。**2026年時点の公式推奨は Databricks Apps 方式**です。

| 観点 | 旧: Model Serving 方式（`old/2_RAGエージェントの構築.ipynb`） | 新: Databricks Apps 方式（本ノートブック） |
|---|---|---|
| エージェント定義 | `ChatAgent` クラス（`predict`/`predict_stream`） | モジュール関数 `@invoke()` / `@stream()` |
| ロギング/登録 | `log_model` → `register_model`（UC） | 不要（コードを直接デプロイ） |
| デプロイ手段 | `agents.deploy()`（Python SDK） | `databricks bundle deploy` → `run`（CLI / DABs） |
| サービング先 | Model Serving エンドポイント | Databricks App（`/responses`） |
| クエリ | `/serving-endpoints/{name}/invocations` | `POST <app-url>/responses` |

> 💡 旧方式（Model Serving）も引き続き動作しますが、[公式ドキュメント](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/deploy-agent)は新規ユースケースでは Apps 方式を推奨しています。旧手順は `old/2_RAGエージェントの構築.ipynb` に残しています。

## このノートブックで学習する内容

1. **MLflow AgentServer / ResponsesAgent** のアーキテクチャ
2. **`@invoke()` / `@stream()`** によるエージェント実装（LangGraph 統合）
3. **Databricks Asset Bundles (DABs)** によるプロジェクト構成（`databricks.yml` / `app.yaml`）
4. **ローカルテスト**（`/invocations`）と **デプロイ**（`databricks bundle`）
5. デプロイした App への**クエリ**（`/responses`）

## 実行環境・前提

- **サーバーレスコンピュート**での実行を想定
- `1_PDFのパースとベクトルインデックスの作成.ipynb` で AI Search Index が作成済み
- **Databricks CLI `>=0.283.0`** がインストール済み（`databricks bundle` を使うため）

## 参考リンク

- [Databricks Apps でエージェントをオーサリング・デプロイ（公式・推奨）](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/author-agent)
- [Model Serving から Apps への移行ガイド](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/migrate-agent-to-apps)
- [app-templates リポジトリ](https://github.com/databricks/app-templates)


## 1. アーキテクチャ：MLflow AgentServer と ResponsesAgent

Databricks Apps 方式では、エージェント本体が **MLflow `AgentServer`（FastAPI ベースの非同期サーバー）** として動作します。Model Serving エンドポイントは作成されず、**アプリ自体がエージェントのサービング実体**になります。

エージェントのロジックは、次の2つのモジュールレベル関数として実装します（クラスではなく**関数**である点が旧方式との大きな違いです）。

- **`@stream()`** … ストリーミング応答を返す非同期ジェネレータ。ロジックの本体をここに集約し、`ResponsesAgentStreamEvent` を `yield` します。
- **`@invoke()`** … 非ストリーミング応答。通常は `@stream()` を呼び出して `response.output_item.done` イベントだけ収集し、`ResponsesAgentResponse` にまとめて返します。

入出力は **OpenAI Responses API 形式**（`input` 配列）です。LangGraph の messages 形式との相互変換には MLflow のヘルパー（`to_chat_completions_input` など）を使います。

### プロジェクトのファイル構成

このノートブックでは、以下のファイルを UC Volume 上のアプリディレクトリに書き出し、DABs でデプロイします。

```
<app_dir>/
├── app.yaml                  # Databricks Apps 起動設定
├── databricks.yml            # DABs バンドル定義（リソース宣言）
├── pyproject.toml            # 依存関係（uv）
└── agent_server/
    ├── __init__.py
    ├── agent.py              # ★エージェント本体（@invoke / @stream）
    ├── utils.py              # LangGraph → Responses イベント変換ヘルパー
    └── start_server.py       # AgentServer 起動エントリポイント
```


## 2. ライブラリの準備

ローカル（このノートブック）でエージェントロジックを検証するためのライブラリをインストールします。デプロイ時にアプリコンテナ側が使う依存は、後述の `pyproject.toml` で別途宣言します。

> 📌 バージョンは公式テンプレート（app-templates）に準拠しています。

In [ ]:
# エージェント実装・ローカル検証用ライブラリ
%pip install -U -qqqq "mlflow>=3.10.0" "databricks-agents>=1.9.3" "databricks-langchain>=0.17.0" "langgraph>=1.1.0"
dbutils.library.restartPython()

## 3. パラメータ設定（widget）

環境依存の値を widget で指定します。上部の入力欄で自分の環境に合わせて変更し、このセルを実行してください。

In [ ]:
# パラメータ設定（widget から取得）
dbutils.widgets.text("CATALOG_NAME", "skato", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_workshop", "スキーマ名")
dbutils.widgets.text("VOLUME_NAME", "pdf_files", "ボリューム名（アプリのソース配置先にも使用）")
dbutils.widgets.text("VECTOR_INDEX_NAME", "chunked_document_vs_index", "AI Search Index名")
dbutils.widgets.text("LLM_ENDPOINT_NAME", "databricks-claude-sonnet-4-5", "LLMエンドポイント名")
dbutils.widgets.text("APP_NAME", "rag-agent-app", "Databricks App 名（英数字とハイフン）")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
VOLUME_NAME = dbutils.widgets.get("VOLUME_NAME")
VECTOR_INDEX_NAME = dbutils.widgets.get("VECTOR_INDEX_NAME")
LLM_ENDPOINT_NAME = dbutils.widgets.get("LLM_ENDPOINT_NAME")
APP_NAME = dbutils.widgets.get("APP_NAME")

# AI Search Index のフルネームと、アプリのソースを置くディレクトリ
VS_INDEX_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{VECTOR_INDEX_NAME}"
APP_SOURCE_DIR = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}/{APP_NAME}"

print(f"AI Search Index: {VS_INDEX_FULLNAME}")
print(f"LLMエンドポイント: {LLM_ENDPOINT_NAME}")
print(f"アプリ名: {APP_NAME}")
print(f"アプリソースディレクトリ: {APP_SOURCE_DIR}")

## 4. アプリのソースディレクトリを作成

エージェントのソースコードを配置するディレクトリを UC Volume 上に作成します。以降のセルで、このディレクトリに各ファイルを書き出します。

In [ ]:
import os

# アプリソース用ディレクトリと agent_server サブディレクトリを作成
os.makedirs(f"{APP_SOURCE_DIR}/agent_server", exist_ok=True)
print(f"作成しました: {APP_SOURCE_DIR}/agent_server")

## 5. エージェント本体（`agent_server/agent.py`）

エージェントのロジックを `@invoke()` / `@stream()` 関数として実装します。ここが Model Serving 方式の `ChatAgent` クラスに代わる中心部です。

**このコードのポイント:**
- `from mlflow.genai.agent_server import invoke, stream` でデコレータを import
- `@stream()` にロジック本体を書き、`@invoke()` はそれを呼んで結果を集約
- `create_agent`（LangChain）で LLM + retriever ツールを束ねた LangGraph エージェントを構築
- `request.input`（Responses 形式）→ `to_chat_completions_input(...)` で LangChain messages に変換
- `agent.astream(...)` の出力を `process_agent_astream_events(...)`（`utils.py`）で `ResponsesAgentStreamEvent` に変換

> 💡 retriever ツールは `1_...` で作成した AI Search Index を参照する `VectorSearchRetrieverTool` を使います。index 名・LLM 名は widget 値をコードに埋め込みます。

In [ ]:
# agent.py を組み立てて書き出す（widget 値をプレースホルダ置換で埋め込む）
AGENT_PY = """# Databricks Apps 版 Tool-calling RAG エージェント（ResponsesAgent / AgentServer）
# - @invoke() / @stream() でエージェントを定義（Model Serving 版の ChatAgent クラスに相当）
# - LangGraph（create_agent）で LLM + AI Search retriever ツールを束ねる
import logging
from typing import AsyncGenerator

import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent
from mlflow.genai.agent_server import invoke, stream
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    to_chat_completions_input,
)

from agent_server.utils import process_agent_astream_events

logger = logging.getLogger(__name__)
mlflow.langchain.autolog()

# システムプロンプト
SYSTEM_PROMPT = (
    "あなたは生成AI開発に関する専門的なアシスタントです。"
    "登録された AI Search Index を活用し、検索された文書の内容に基づいて、"
    "日本語で丁寧かつ正確に回答してください。不確かな情報は推測しないでください。"
)


def init_agent():
    # AI Search Index を参照する retriever ツール
    retriever_tool = VectorSearchRetrieverTool(
        index_name="__VS_INDEX_FULLNAME__",
        tool_description=(
            "生成AI開発に関する技術文書やベストプラクティスを検索するツール。"
            "GenAI開発ワークフロー、MLOps、エージェント設計などの質問に使用する。"
        ),
    )
    llm = ChatDatabricks(endpoint="__LLM_ENDPOINT_NAME__")
    return create_agent(model=llm, tools=[retriever_tool], system_prompt=SYSTEM_PROMPT)


@stream()
async def stream_handler(request: ResponsesAgentRequest) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    agent = init_agent()
    messages = {"messages": to_chat_completions_input([i.model_dump() for i in request.input])}
    async for event in process_agent_astream_events(
        agent.astream(input=messages, stream_mode=["updates", "messages"])
    ):
        yield event


@invoke()
async def invoke_handler(request: ResponsesAgentRequest) -> ResponsesAgentResponse:
    # @stream() の結果から完了アイテムだけを集めて返す
    outputs = [
        event.item
        async for event in stream_handler(request)
        if event.type == "response.output_item.done"
    ]
    return ResponsesAgentResponse(output=outputs)
"""

AGENT_PY = AGENT_PY.replace("__VS_INDEX_FULLNAME__", VS_INDEX_FULLNAME).replace("__LLM_ENDPOINT_NAME__", LLM_ENDPOINT_NAME)
with open(f"{APP_SOURCE_DIR}/agent_server/agent.py", "w", encoding="utf-8") as f:
    f.write(AGENT_PY)
print(f"書き出しました: {APP_SOURCE_DIR}/agent_server/agent.py")
print(AGENT_PY)

## 6. ストリーム変換ヘルパー（`agent_server/utils.py`）

LangGraph の `astream`（`updates` / `messages`）の出力を、Responses API の `ResponsesAgentStreamEvent` に変換するヘルパーです。公式テンプレート（app-templates）の実装に準拠しています。

- `updates` イベント → ノードが生成したメッセージを `output_to_responses_items_stream(...)` で変換
- `messages` イベント → LLM のトークン差分（`AIMessageChunk`）を `create_text_delta(...)` でテキストデルタとして流す

In [ ]:
# utils.py を書き出す（widget 依存なし）
UTILS_PY = """# LangGraph の astream 出力を Responses イベントに変換するヘルパー
import json
import logging
from typing import Any, AsyncGenerator, AsyncIterator

from langchain.messages import AIMessageChunk, ToolMessage
from mlflow.types.responses import (
    ResponsesAgentStreamEvent,
    create_text_delta,
    output_to_responses_items_stream,
)


async def process_agent_astream_events(
    async_stream: AsyncIterator[Any],
) -> AsyncGenerator[ResponsesAgentStreamEvent, None]:
    async for event in async_stream:
        if event[0] == "updates":
            for node_data in event[1].values():
                if len(node_data.get("messages", [])) > 0:
                    for msg in node_data["messages"]:
                        # ツール結果が非文字列なら JSON 文字列化
                        if isinstance(msg, ToolMessage) and not isinstance(msg.content, str):
                            msg.content = json.dumps(msg.content)
                    for item in output_to_responses_items_stream(node_data["messages"]):
                        yield item
        elif event[0] == "messages":
            try:
                chunk = event[1][0]
                if isinstance(chunk, AIMessageChunk) and (content := chunk.content):
                    yield ResponsesAgentStreamEvent(
                        **create_text_delta(delta=content, item_id=chunk.id)
                    )
            except Exception as e:
                logging.exception(f"Error processing agent stream event: {e}")
"""

with open(f"{APP_SOURCE_DIR}/agent_server/utils.py", "w", encoding="utf-8") as f:
    f.write(UTILS_PY)
# __init__.py（空パッケージマーカー）も作成
open(f"{APP_SOURCE_DIR}/agent_server/__init__.py", "w").close()
print("書き出しました: agent_server/utils.py, agent_server/__init__.py")

## 7. サーバー起動エントリポイント（`agent_server/start_server.py`）

`AgentServer` を初期化し、`agent.py` の `@invoke`/`@stream` を登録して起動するエントリポイントです。通常このファイルは編集不要で、テンプレートそのままです。

In [ ]:
# start_server.py を書き出す
START_SERVER_PY = """# AgentServer 起動エントリポイント（通常は編集不要）
from pathlib import Path

from dotenv import load_dotenv
from mlflow.genai.agent_server import AgentServer, setup_mlflow_git_based_version_tracking

# .env があれば読み込む（ローカル実行時の認証用）
load_dotenv(dotenv_path=Path(__file__).parent.parent / ".env", override=True)

# agent モジュールを import して @invoke / @stream 関数をサーバーに登録
import agent_server.agent  # noqa: E402

agent_server = AgentServer("ResponsesAgent", enable_chat_proxy=True)

# 複数ワーカー対応のため app をモジュールレベル変数として公開
app = agent_server.app  # noqa: F841
setup_mlflow_git_based_version_tracking()


def main():
    agent_server.run(app_import_string="agent_server.start_server:app")
"""

with open(f"{APP_SOURCE_DIR}/agent_server/start_server.py", "w", encoding="utf-8") as f:
    f.write(START_SERVER_PY)
print("書き出しました: agent_server/start_server.py")

## 8. 依存関係定義（`pyproject.toml`）

アプリコンテナ側の依存関係を宣言します。Databricks Apps は `uv` を使うため `pyproject.toml`（＋ `uv.lock`）が推奨です。`[project.scripts]` の `start-server` が起動コマンドの実体になります。

In [ ]:
# pyproject.toml を書き出す
PYPROJECT_TOML = """[project]
name = "rag-agent-server"
version = "0.1.0"
description = "RAG agent on Databricks Apps (ResponsesAgent)"
readme = "README.md"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.129.0",
    "uvicorn>=0.41.0",
    "mlflow>=3.10.0",
    "databricks-agents>=1.9.3",
    "databricks-langchain>=0.17.0",
    "langgraph>=1.1.0",
    "python-dotenv>=1.2.1",
    "opentelemetry-exporter-otlp-proto-grpc>=1.25.0",
]

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project.scripts]
start-server = "agent_server.start_server:main"
"""

with open(f"{APP_SOURCE_DIR}/pyproject.toml", "w", encoding="utf-8") as f:
    f.write(PYPROJECT_TOML)
print("書き出しました: pyproject.toml")

## 9. Apps 起動設定（`app.yaml`）

Databricks Apps がコンテナ起動時に実行するコマンドと環境変数を定義します。ここでは backend（AgentServer）のみを起動する構成にします。

> ⚠️ `app.yaml` のリソース参照は `valueFrom`（camelCase）、後述の `databricks.yml` の `config.env` は `value_from`（snake_case）です。混同しないよう注意してください。

In [ ]:
# app.yaml を書き出す
APP_YAML = """command: ["uv", "run", "start-server"]

env:
  - name: MLFLOW_TRACKING_URI
    value: "databricks"
  - name: MLFLOW_REGISTRY_URI
    value: "databricks-uc"
"""

with open(f"{APP_SOURCE_DIR}/app.yaml", "w", encoding="utf-8") as f:
    f.write(APP_YAML)
print("書き出しました: app.yaml")

## 10. DABs バンドル定義（`databricks.yml`）

Databricks Asset Bundles の定義です。アプリ本体と、エージェントが**アクセスするリソース**（LLM サービングエンドポイント・AI Search Index）を宣言します。

**リソース宣言の対応:**
- LLM の Serving Endpoint → `serving_endpoint`（`CAN_QUERY`）
- AI Search Index → `uc_securable`（`securable_type: TABLE`, `SELECT`）

> 📌 `bundle deploy` はファイルアップロードとリソース構成のみで、**`bundle run` を実行しないとアプリは新コードで起動しません**。

In [ ]:
# databricks.yml を widget 値で組み立てて書き出す
DATABRICKS_YML = """bundle:
  name: rag_agent_bundle

resources:
  apps:
    rag_agent_app:
      name: "__APP_NAME__"
      description: "RAG agent (ResponsesAgent) on Databricks Apps"
      source_code_path: ./
      config:
        command: ["uv", "run", "start-server"]
        env:
          - name: MLFLOW_TRACKING_URI
            value: "databricks"
          - name: MLFLOW_REGISTRY_URI
            value: "databricks-uc"

      # このアプリがアクセスできる Databricks リソース
      resources:
        - name: "llm_endpoint"
          serving_endpoint:
            name: "__LLM_ENDPOINT_NAME__"
            permission: "CAN_QUERY"
        - name: "vector_index"
          uc_securable:
            securable_full_name: "__VS_INDEX_FULLNAME__"
            securable_type: "TABLE"
            permission: "SELECT"

targets:
  dev:
    mode: development
    default: true
"""

DATABRICKS_YML = (DATABRICKS_YML
    .replace("__APP_NAME__", APP_NAME)
    .replace("__LLM_ENDPOINT_NAME__", LLM_ENDPOINT_NAME)
    .replace("__VS_INDEX_FULLNAME__", VS_INDEX_FULLNAME))
with open(f"{APP_SOURCE_DIR}/databricks.yml", "w", encoding="utf-8") as f:
    f.write(DATABRICKS_YML)
print(f"書き出しました: {APP_SOURCE_DIR}/databricks.yml")
print(DATABRICKS_YML)

## 11. （任意）ノートブック内でのエージェント動作確認

デプロイ前に、エージェントのロジックをこのノートブック内で簡易的に確認できます。`init_agent()` を呼んで LangGraph エージェントを直接実行します。

> ℹ️ 本格的なローカルサーバー検証（`uv run start-server` → `POST /invocations`）はターミナル環境で行います（下のマークダウン参照）。ここではグラフ実行だけを確認します。

In [ ]:
# agent_server パッケージを import 可能にしてグラフを直接実行（任意）
import sys
if APP_SOURCE_DIR not in sys.path:
    sys.path.insert(0, APP_SOURCE_DIR)

from agent_server.agent import init_agent

agent = init_agent()
result = agent.invoke({"messages": [{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}]})

for msg in result["messages"]:
    role = getattr(msg, "type", "?")
    content = getattr(msg, "content", "")
    if content:
        print(f"[{role}] {content[:300]}")

### 参考：ターミナルでのローカルサーバー検証

アプリのソースを手元に取得できる環境では、以下でローカルの AgentServer を起動して疎通確認できます（**ローカルは `/invocations`**、デプロイ後は `/responses`）。

```bash
cd <app_dir>
uv sync                       # 依存解決 + uv.lock 生成
uv run start-server --reload  # http://localhost:8000 で起動

# 別ターミナルから
curl -X POST http://localhost:8000/invocations \
  -H "Content-Type: application/json" \
  -d '{ "input": [{ "role": "user", "content": "こんにちは" }], "stream": true }'
```


## 12. Databricks Apps へのデプロイ（DABs / CLI）

`databricks bundle` コマンドでデプロイします。ノートブックのシェルから実行できますが、認証プロファイルの設定によってはターミナルでの実行が確実です。

**デプロイの順序:**
1. `databricks bundle validate` — バンドル構成の検証
2. `databricks bundle deploy` — コードアップロード + リソース構成
3. `databricks bundle run rag_agent_app` — **アプリの起動/再起動（必須）**

> ⚠️ `bundle run` を実行しないとアプリは新しいコードで起動しません。`rag_agent_app` は `databricks.yml` の `resources.apps` 配下のキー名です。

In [ ]:
# databricks bundle コマンドを APP_SOURCE_DIR で実行するヘルパー
import subprocess

def run_cli(args):
    print(f"$ {' '.join(args)}  (cwd={APP_SOURCE_DIR})")
    proc = subprocess.run(args, cwd=APP_SOURCE_DIR, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print("--- STDERR ---")
        print(proc.stderr)
    return proc.returncode

# 1) バンドル構成の検証
run_cli(["databricks", "bundle", "validate"])

In [ ]:
# 2) デプロイ（コードアップロード + リソース構成）
run_cli(["databricks", "bundle", "deploy"])

In [ ]:
# 3) アプリの起動/再起動（必須）
run_cli(["databricks", "bundle", "run", "rag_agent_app"])

## 13. デプロイした App へのクエリ

デプロイ後、App の URL に対して **`POST /responses`** でクエリします。リクエストボディは OpenAI Responses API 形式（`input` 配列）です。

**認証**: App へのクエリは **OAuth トークン**を使います（PAT は不可）。トークンは以下で取得します。

```bash
databricks auth token | jq -r '.access_token'
```

### curl での例

```bash
curl -X POST "https://<app-url>.databricksapps.com/responses" \
  -H "Authorization: Bearer <oauth-token>" \
  -H "Content-Type: application/json" \
  -d '{ "input": [{ "role": "user", "content": "生成AIの開発ワークフローについて教えてください" }], "stream": true }'
```

### Python（Databricks OpenAI クライアント）での例

`databricks-openai` を使うと、App 名を指定するだけで OAuth 認証込みで呼び出せます。

In [ ]:
# Databricks OpenAI クライアントをインストール
%pip install -U -qqqq databricks-openai
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

# restartPython 後にこのセルから実行される場合に備えて widget を再取得
APP_NAME = dbutils.widgets.get("APP_NAME")

w = WorkspaceClient()
client = DatabricksOpenAI(workspace_client=w)

# 非ストリーミングでクエリ（model は "apps/<app-name>" 形式）
response = client.responses.create(
    model=f"apps/{APP_NAME}",
    input=[{"role": "user", "content": "生成AIの開発ワークフローについて教えてください"}],
)
print(response)

## 14. まとめと次のステップ

### このノートブックで学んだこと

- **Databricks Apps 方式**でのエージェント構築（`@invoke()` / `@stream()` + `AgentServer`）
- LangGraph エージェントの Responses API 形式への統合（`utils.py` の変換ヘルパー）
- **DABs**（`databricks.yml` / `app.yaml` / `pyproject.toml`）によるプロジェクト構成とリソース宣言
- `databricks bundle validate → deploy → run` によるデプロイ
- デプロイした App への `POST /responses` クエリ（OAuth 認証）

### 旧方式との違い（振り返り）

| | 旧: Model Serving | 新: Databricks Apps |
|---|---|---|
| 実装 | `ChatAgent` クラス | `@invoke`/`@stream` 関数 |
| デプロイ | `agents.deploy()` | `databricks bundle` |
| サービング | Model Serving エンドポイント | Databricks App |

Model Serving 方式の手順は `old/2_RAGエージェントの構築.ipynb` に残しています。

### 次のステップ

1. **評価**: 姉妹ノートブック（MLflow 評価詳細版）で、合成データ生成・カスタムスコアラー・LLM-as-a-Judge・プロンプトレジストリを学ぶ
2. **モニタリング**: MLflow 3 のリアルタイムトレーシングでエージェントの挙動を追跡
3. **UI 提供**: `3_Webアプリケーションのデプロイ.ipynb` の Streamlit チャットUIから、この App の `/responses` を呼び出す

### 参考リンク

- [Databricks Apps でエージェントをオーサリング・デプロイ](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/author-agent)
- [Model Serving から Apps への移行ガイド](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/migrate-agent-to-apps)
- [app-templates リポジトリ](https://github.com/databricks/app-templates)
- [Databricks Asset Bundles](https://docs.databricks.com/aws/en/dev-tools/bundles/)
